In [11]:
from pathlib import Path
import os

from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/ML_project_dataset'
OUT_DIR = DATA_DIR
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

for name in ['train.csv', 'test.csv', 'submission.csv']:
    path = Path(DATA_DIR) / name
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Update DATA_DIR to your dataset folder.')
print('Data directory:', DATA_DIR)
print('Output directory:', OUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data directory: /content/drive/MyDrive/ML_project_dataset
Output directory: /content/drive/MyDrive/ML_project_dataset


In [15]:
"""Flight price model: block-neighbour features + HistGradientBoosting blend. Needs only pandas/numpy/scikit-learn.
Set D to the folder containing train.csv, test.csv, submission.csv. Writes submission.csv."""
import numpy as np, pandas as pd

import os
D = DATA_DIR
STOPS = {"zero": 0, "one": 1, "two_or_more": 2}

def load():
    tr = pd.read_csv(f"{D}/train.csv"); te = pd.read_csv(f"{D}/test.csv")
    al = pd.concat([tr.assign(is_train=1), te.assign(is_train=0, price=np.nan)]).sort_values("id").reset_index(drop=True)
    key = ["source_city", "destination_city", "flight_class", "days_left"]
    al["seg"] = (al[key] != al[key].shift()).any(axis=1).cumsum()
    al["pos"] = al.groupby("seg").cumcount()
    al["seglen"] = al.groupby("seg").id.transform("size")
    al["relpos"] = (al.pos + 0.5) / al.seglen
    al["stops_n"] = al.stops.map(STOPS)
    al["route"] = al.source_city + "_" + al.destination_city
    for c in ["airline", "source_city", "destination_city", "departure_time", "arrival_time", "flight_class", "route", "flight"]:
        al[c + "_c"] = al[c].astype("category").cat.codes
    return tr, te, al

K = 3

def neighbor_feats(al, known):
    """known: boolean array (len(al)) of rows whose price may be used. Self is always excluded."""
    n = len(al)
    price = al.price.values
    seg = al.seg.values
    pos = al.pos.values
    stops = al.stops_n.values
    dur = al.duration.values
    airl = al.airline_c.values
    flt = al.flight_c.values
    out = {}
    for k in range(1, K + 1):
        for s in "pn":
            out[f"{s}{k}"] = np.full(n, np.nan); out[f"{s}{k}_d"] = np.full(n, np.nan)
            out[f"{s}{k}_stops"] = np.full(n, np.nan); out[f"{s}{k}_dur"] = np.full(n, np.nan)
            out[f"{s}{k}_sameair"] = np.full(n, np.nan); out[f"{s}{k}_sameflt"] = np.full(n, np.nan)
    for c in ["sg_cnt", "sg_mean", "sg_med", "sg_min", "sg_max", "sg_std", "sg_rank_frac", "sg_nlower", "sg_nhigher"]:
        out[c] = np.full(n, np.nan)
    starts = np.r_[0, np.flatnonzero(np.diff(seg)) + 1, n]
    for a, b in zip(starts[:-1], starts[1:]):
        idx = np.arange(a, b)
        kn = idx[known[a:b]]
        kp = price[kn]
        for i in idx:
            # known rows excluding self
            m = kn != i
            ki = kn[m]
            if len(ki) == 0:
                continue
            before = ki[ki < i][::-1]
            after = ki[ki > i]
            for k in range(1, K + 1):
                if len(before) >= k:
                    j = before[k - 1]
                    out[f"p{k}"][i] = price[j]; out[f"p{k}_d"][i] = i - j
                    out[f"p{k}_stops"][i] = stops[j] - stops[i]; out[f"p{k}_dur"][i] = dur[j] - dur[i]
                    out[f"p{k}_sameair"][i] = airl[j] == airl[i]; out[f"p{k}_sameflt"][i] = flt[j] == flt[i]
                if len(after) >= k:
                    j = after[k - 1]
                    out[f"n{k}"][i] = price[j]; out[f"n{k}_d"][i] = j - i
                    out[f"n{k}_stops"][i] = stops[j] - stops[i]; out[f"n{k}_dur"][i] = dur[j] - dur[i]
                    out[f"n{k}_sameair"][i] = airl[j] == airl[i]; out[f"n{k}_sameflt"][i] = flt[j] == flt[i]
            pr = price[ki]
            out["sg_cnt"][i] = len(ki); out["sg_mean"][i] = pr.mean(); out["sg_med"][i] = np.median(pr)
            out["sg_min"][i] = pr.min(); out["sg_max"][i] = pr.max(); out["sg_std"][i] = pr.std()
            out["sg_nlower"][i] = len(before); out["sg_nhigher"][i] = len(after)
            out["sg_rank_frac"][i] = len(before) / len(ki)
    F = pd.DataFrame(out)
    # linear interpolation between nearest known neighbours (by position)
    w = F.p1_d / (F.p1_d + F.n1_d)
    F["interp"] = F.p1 + w * (F.n1 - F.p1)
    F["interp"] = F.interp.fillna(F.p1).fillna(F.n1)
    F["nb_mean"] = F[["p1", "n1"]].mean(axis=1)
    F["gap"] = F.n1 - F.p1
    F["interp_all"] = F.interp.fillna(F.sg_med)
    return F

BASE = ["duration", "days_left", "stops_n", "relpos", "pos", "seglen", "airline_c", "source_city_c", "destination_city_c",
        "departure_time_c", "arrival_time_c", "flight_class_c", "route_c"]


def airline_feats(al, known):
    """same-airline / same-airline+stops / same-flight neighbour statistics inside the block (self excluded)."""
    n = len(al); price = al.price.values; seg = al.seg.values
    airl = al.airline_c.values; stops = al.stops_n.values; flt = al.flight_c.values
    cols = ["sa_p1", "sa_p1_d", "sa_n1", "sa_n1_d", "sa_mean", "sa_cnt", "sa_min", "sa_max", "sa_med",
            "sas_p1", "sas_n1", "sas_mean", "sas_cnt", "sf_mean", "sf_cnt", "sa_other_mean",
            "sa_p2", "sa_n2", "sas_p1_d", "sas_n1_d"]
    out = {c: np.full(n, np.nan) for c in cols}
    starts = np.r_[0, np.flatnonzero(np.diff(seg)) + 1, n]
    for a, b in zip(starts[:-1], starts[1:]):
        idx = np.arange(a, b); kn = idx[known[a:b]]
        for i in idx:
            ki = kn[kn != i]
            if len(ki) == 0: continue
            same = ki[airl[ki] == airl[i]]
            if len(same):
                pr = price[same]
                bf = same[same < i]; af = same[same > i]
                if len(bf): out["sa_p1"][i] = price[bf[-1]]; out["sa_p1_d"][i] = i - bf[-1]
                if len(af): out["sa_n1"][i] = price[af[0]]; out["sa_n1_d"][i] = af[0] - i
                if len(bf) > 1: out["sa_p2"][i] = price[bf[-2]]
                if len(af) > 1: out["sa_n2"][i] = price[af[1]]
                out["sa_mean"][i] = pr.mean(); out["sa_cnt"][i] = len(same); out["sa_min"][i] = pr.min()
                out["sa_max"][i] = pr.max(); out["sa_med"][i] = np.median(pr)
                ss = same[stops[same] == stops[i]]
                if len(ss):
                    bf = ss[ss < i]; af = ss[ss > i]
                    if len(bf): out["sas_p1"][i] = price[bf[-1]]; out["sas_p1_d"][i] = i - bf[-1]
                    if len(af): out["sas_n1"][i] = price[af[0]]; out["sas_n1_d"][i] = af[0] - i
                    out["sas_mean"][i] = price[ss].mean(); out["sas_cnt"][i] = len(ss)
                sf = same[flt[same] == flt[i]]
                if len(sf): out["sf_mean"][i] = price[sf].mean(); out["sf_cnt"][i] = len(sf)
            oth = ki[airl[ki] != airl[i]]
            if len(oth): out["sa_other_mean"][i] = price[oth].mean()
    F = pd.DataFrame(out)
    F["sa_interp"] = np.where(F.sa_p1.notna() & F.sa_n1.notna(),
                              F.sa_p1 + (F.sa_p1_d / (F.sa_p1_d + F.sa_n1_d)) * (F.sa_n1 - F.sa_p1),
                              F.sa_p1.fillna(F.sa_n1))
    F["sa_eq"] = (F.sa_p1 == F.sa_n1).astype(float)
    F.loc[F.sa_p1.isna() | F.sa_n1.isna(), "sa_eq"] = np.nan
    return F

def all_feats(al, known):
    F = pd.concat([neighbor_feats(al, known), airline_feats(al, known)], axis=1)
    F["from_end"] = (al.seglen - 1 - al.pos).values
    F["is_last"] = (F.from_end == 0).astype(float)
    return F


def cross_feats(al, known):
    """price of the same airline/route/class/stops (and same flight) on nearby days, excluding the row's own block."""
    n = len(al); price = al.price.values; seg = al.seg.values; dl = al.days_left.values
    out = {c: np.full(n, np.nan) for c in ["g1_w1", "g1_w3", "g1_all", "g1_cnt3", "g1_dm", "g1_dp", "g2_w1", "g2_w3", "g2_all", "g2_cnt3", "g2_dm", "g2_dp"]}
    specs = {"g1": ["airline", "route", "flight_class", "stops"], "g2": ["flight", "route", "flight_class"]}
    for name, keys in specs.items():
        gid = al.groupby(keys).ngroup().values
        order = np.argsort(gid, kind="stable")
        gs = np.r_[0, np.flatnonzero(np.diff(gid[order])) + 1, n]
        for a, b in zip(gs[:-1], gs[1:]):
            idx = order[a:b]
            kidx = idx[known[idx]]
            if len(kidx) == 0: continue
            kd = dl[kidx]; kp = price[kidx]; ks = seg[kidx]
            for i in idx:
                m = ks != seg[i]
                if not m.any(): continue
                d = np.abs(kd[m] - dl[i]); p = kp[m]
                out[f"{name}_all"][i] = p.mean()
                w = d <= 1
                if w.any(): out[f"{name}_w1"][i] = p[w].mean()
                w = d <= 3
                if w.any(): out[f"{name}_w3"][i] = p[w].mean(); out[f"{name}_cnt3"][i] = w.sum()
                dd = kd[m] - dl[i]
                q = dd < 0
                if q.any(): out[f"{name}_dm"][i] = p[q][np.argmax(dd[q])]
                q = dd > 0
                if q.any(): out[f"{name}_dp"][i] = p[q][np.argmin(dd[q])]
    return pd.DataFrame(out)

def all_feats2(al, known):
    return pd.concat([all_feats(al, known), cross_feats(al, known)], axis=1)



In [16]:
from sklearn.ensemble import HistGradientBoostingRegressor as H

tr, te, al = load()
sample = pd.read_csv(f"{D}/submission.csv")
known = (al.is_train == 1).values
F = all_feats2(al, known)
X = pd.concat([al[BASE].reset_index(drop=True), F], axis=1)
trn = (al.is_train == 1) & (al.price < 5000)  # exclude these extreme fares from model fitting only
Xtr, ytr = X[trn.values], al.price[trn].values
Xte = X[(al.is_train == 0).values]
ids = al.id[(al.is_train == 0)].values

cfgs = [
    dict(max_iter=800, learning_rate=0.03, max_leaf_nodes=48, min_samples_leaf=8, l2_regularization=5.0, max_features=0.4),
    dict(max_iter=1000, learning_rate=0.025, max_leaf_nodes=64, min_samples_leaf=5, l2_regularization=8.0, max_features=0.3),
    dict(max_iter=800, learning_rate=0.03, max_leaf_nodes=48, min_samples_leaf=8, l2_regularization=5.0, max_features=0.25),
]
preds = []
for ci, cfg in enumerate(cfgs):
    for seed in (0, 1):
        model = H(random_state=100 * ci + seed, **cfg)
        preds.append(model.fit(Xtr, ytr).predict(Xte))
        print(f'V1 fitted config {ci+1}/3, seed {seed}')

p = np.clip(np.mean(preds, axis=0), 0, None)
v1_sub = sample.copy()
v1_sub['price'] = pd.Series(p, index=ids).reindex(v1_sub.id).values
assert v1_sub.price.notna().all() and len(v1_sub) == len(te) and set(v1_sub.id) == set(te.id)
V1_PATH = f'{OUT_DIR}/submission_v1_neighbor_model.csv'
v1_sub.to_csv(V1_PATH, index=False)
print('Saved V1 submission:', V1_PATH)
display(v1_sub.price.describe().to_frame('price').round(2))


V1 fitted config 1/3, seed 0
V1 fitted config 1/3, seed 1
V1 fitted config 2/3, seed 0
V1 fitted config 2/3, seed 1
V1 fitted config 3/3, seed 0
V1 fitted config 3/3, seed 1
Saved V1 submission: /content/drive/MyDrive/ML_project_dataset/submission_v1_neighbor_model.csv


,price
count,4520.00
mean,279.57
std,198.74
min,49.70
25%,158.28
50%,234.02
75%,334.59
max,2161.23


In [17]:
from sklearn.ensemble import HistGradientBoostingRegressor as H, ExtraTreesRegressor

# Reuse the same loaded data and shared features; no external .py import is required.
known = (al.is_train == 1).values
F2 = all_feats2(al, known)
X2 = pd.concat([al[BASE].reset_index(drop=True), F2], axis=1)
is_last = (F2.from_end == 0).values
trn_last = is_last & (al.is_train == 1).values & (al.price.values < 5000)
tst_last = is_last & (al.is_train == 0).values
base = F2.p1.fillna(F2.sg_max).fillna(280).clip(lower=20).values
y_all = al.price.values
print('Last-row training rows:', int(trn_last.sum()), '| test last rows:', int(tst_last.sum()))

P = []
for seed in (0, 1, 2):
    model = H(max_iter=300, learning_rate=0.04, max_leaf_nodes=8, min_samples_leaf=15, l2_regularization=5, max_features=0.5, random_state=seed)
    P.append(('raw', model.fit(X2[trn_last], y_all[trn_last]).predict(X2[tst_last])))

    model = H(max_iter=300, learning_rate=0.04, max_leaf_nodes=8, min_samples_leaf=15, l2_regularization=5, max_features=0.5, random_state=seed)
    ratio_pred = np.exp(model.fit(X2[trn_last], np.log((y_all[trn_last] + 20) / (base[trn_last] + 20))).predict(X2[tst_last])) * (base[tst_last] + 20) - 20
    P.append(('hgb_ratio', ratio_pred))

    model = ExtraTreesRegressor(n_estimators=150, min_samples_leaf=3, max_features=0.5, random_state=seed, n_jobs=1)
    ratio_pred = np.exp(model.fit(X2[trn_last], np.log((y_all[trn_last] + 20) / (base[trn_last] + 20))).predict(X2[tst_last])) * (base[tst_last] + 20) - 20
    P.append(('et_ratio', ratio_pred))
    print(f'V2 fitted seed {seed}')

kinds = {name: np.mean([pred for kind, pred in P if kind == name], axis=0) for name in ('raw', 'hgb_ratio', 'et_ratio')}
last_pred = np.mean(list(kinds.values()), axis=0)

v1_sub = pd.read_csv(V1_PATH)
v2_sub = v1_sub.copy().set_index('id')
ids_last = al.id.values[tst_last]
v2_sub.loc[ids_last, 'price'] = np.clip(0.5 * v1_sub.set_index('id').loc[ids_last, 'price'].values + 0.5 * last_pred, 0, None)
v2_sub = v2_sub.reset_index()
assert (v2_sub.id.values == v1_sub.id.values).all() and v2_sub.price.notna().all() and len(v2_sub) == len(te)
V2_PATH = f'{OUT_DIR}/submission_v2_lastrow_blend.csv'
v2_sub.to_csv(V2_PATH, index=False)
diff = v2_sub.price - v1_sub.price
changed = diff.abs() > 1e-9
print('Saved V2 submission:', V2_PATH)
print('Rows changed:', int(changed.sum()), '| mean absolute change on changed rows:', round(diff.abs()[changed].mean(), 2) if changed.any() else 0, '| max absolute change:', round(diff.abs().max(), 2))
print('V1/V2 prediction correlation:', round(np.corrcoef(v1_sub.price, v2_sub.price)[0, 1], 5))
display(v2_sub.price.describe().to_frame('price').round(2))


Last-row training rows: 1096 | test last rows: 265
V2 fitted seed 0
V2 fitted seed 1
V2 fitted seed 2
Saved V2 submission: /content/drive/MyDrive/ML_project_dataset/submission_v2_lastrow_blend.csv
Rows changed: 265 | mean absolute change on changed rows: 24.8 | max absolute change: 337.41
V1/V2 prediction correlation: 0.99796


,price
count,4520.00
mean,279.55
std,200.13
min,49.70
25%,158.28
50%,233.88
75%,334.17
max,2235.95


In [2]:
import pandas as pd
from IPython.display import display

submission_versions = [
    {"Version": "V1 neighbor-feature model", "File": "submission_v1_neighbor_model.csv", "Test rows": 4520, "Mean predicted price": 279.57, "Median predicted price": 234.02, "Min prediction": 49.70, "Max prediction": 2161.23, "Validation RMSE": None, "Validation MAE": None, "Validation R2": None},
    {"Version": "V2 last-row blend", "File": "submission_v2_lastrow_blend.csv", "Test rows": 4520, "Mean predicted price": 279.55, "Median predicted price": 233.88, "Min prediction": 49.70, "Max prediction": 2235.95, "Validation RMSE": None, "Validation MAE": None, "Validation R2": None},
]
display(pd.DataFrame(submission_versions))
print("V1/V2 prediction correlation recorded in notebook: 0.99796")
print("V2 changed 265 test-row predictions; mean absolute change among changed rows: 24.8.")
# print("Validation metrics are not available in this notebook; do not infer accuracy from prediction summaries.")

,Version,File,Test rows,Mean predicted price,Median predicted price,Min prediction,Max prediction,Validation RMSE,Validation MAE,Validation R2
0,V1 neighbor-feature model,submission_v1_neighbor_model.csv,4520,279.57,234.02,49.7,2161.23,None,None,None
1,V2 last-row blend,submission_v2_lastrow_blend.csv,4520,279.55,233.88,49.7,2235.95,None,None,None


V1/V2 prediction correlation recorded in notebook: 0.99796
V2 changed 265 test-row predictions; mean absolute change among changed rows: 24.8.
